<img src="../data/zls.svg" width="120" style="float:right; margin: 0 20px 10px 20px;">

# 01 — TTS Basics

Goal: send text to the API, wait for the result, decode the returned WAV, and play it.

### Step 1 — Create a session

Every interaction with the API starts with a **session**. Think of it as a temporary workspace on the server.

In [ ]:
import base64
import time
import requests
from IPython.display import Audio

BASE_URL = "https://sproochmaschinn.lu"

In [ ]:
session = requests.post(f"{BASE_URL}/api/session").json()
session_id = session["session_id"]
print("Session created:", session_id)

### Step 2 — Submit a TTS request

We send text and a model name. The API returns a `request_id` — it does **not** return audio immediately. The synthesis happens asynchronously on the server.

In [ ]:
text = "Moien, wéi geet et?"
model = "claude"

tts_response = requests.post(
    f"{BASE_URL}/api/tts/{session_id}",
    json={"text": text, "model": model}
).json()

request_id = tts_response["request_id"]
print("TTS request_id:", request_id)

### Step 3 — Poll for the result

Since TTS is async, we keep checking until the status is `"completed"`. We add a limit so we don't wait forever if something goes wrong.

In [ ]:
MAX_POLLS = 60

for _ in range(MAX_POLLS):
    result = requests.get(f"{BASE_URL}/api/result/{request_id}").json()
    if result["status"] == "completed":
        break
    time.sleep(1)
else:
    raise TimeoutError("TTS request did not complete in time.")

result

### Step 4 — Decode and save the audio

The result contains the audio as a **base64-encoded** string. We decode it back to bytes and write a WAV file.

In [ ]:
audio_b64 = result["result"]["data"]
audio_bytes = base64.b64decode(audio_b64)

with open("tts_output.wav", "wb") as f:
    f.write(audio_bytes)

print("Saved:", "tts_output.wav")

### Step 5 — Listen

In [ ]:
Audio("tts_output.wav")

## Try it yourself

**Easy:** Change the text to `"Wéi ass d'Wieder haut?"` and re-run from Step 2.

**Medium:** Print the `result` dictionary and look at what metadata comes back alongside the audio (e.g. duration, sample rate). How does the metadata change if you use a longer sentence?

**Stretch:** Submit two different texts, save them to different files, and listen to both.